# REST Inference for single-model server deployment 

_Note: Use this procedure for testing a model that you deployed on a single-model server. See `3_rest_requests_multi_model.ipynb` for testing a model that you deployed on a multi-model server._

## First, replace the placeholder with the *url* or *External* you got at the previous step from the Model Serving configuration

The model name is already filled in.

In [1]:
model_name = "transe"
infer_endpoint = 'http://modelmesh-serving.ha25-largegraphmodels-for-cysec-ae498b:8008' # 'http://transe-predictor.ha25-largegraphmodels-for-cysec-ae498b.svc.cluster.local' # e.g. 'https://model-server.<your-namespace>.svc.cluster.local' or 'https://model-server-<your-namespace>.apps.shift.nerc.mghpcc.org'
infer_url = f"{infer_endpoint}/v2/models/{model_name}/infer"

## Request Function

Build and submit the REST request. 

Note: You submit the data in the same format that you used for an ONNX inference.

In [2]:
import requests
import numpy as np

def transe_rest_request(pos_triples, neg_triples):
    """
    pos_triples: np.ndarray of shape (B, 3), dtype int64
    neg_triples: np.ndarray of shape (B, N, 3), dtype int64
    """

    json_data = {
        "inputs": [
            {
                "name": "pos_triples",
                "shape": list(pos_triples.shape),
                "datatype": "INT64",
                "data": pos_triples.tolist()
            },
            {
                "name": "neg_triples",
                "shape": list(neg_triples.shape),
                "datatype": "INT64",
                "data": neg_triples.tolist()
            }
        ]
    }

    response = requests.post(infer_url, json=json_data)
    response.raise_for_status()
    return response.json()


In [3]:
pos = np.array([[0, 0, 1]], dtype=np.int64)
neg = np.array([
    [
        [0, 0, 2],
        [3, 0, 1]
    ]
], dtype=np.int64)

result = transe_rest_request(pos, neg)

print(result)

{'model_name': 'transe__isvc-3459e64133', 'model_version': '1', 'outputs': [{'name': 'neg_score', 'datatype': 'FP32', 'data': [-21.065945]}, {'name': 'pos_score', 'datatype': 'FP32', 'data': [-10.950191]}]}


In [61]:
import pandas as pd
import ast
import pickle
import os
import time
from tqdm.notebook import tqdm

In [6]:
folder_path = "saved/2026-03-23T15-40-42-551Z/cve_metadata_2026-03-23T15:40:42.551Z.csv"
df = pd.read_csv(folder_path)
df["matching_cwes"] = df["matching_cwes"].apply(ast.literal_eval)

In [32]:
df_cwe = pd.read_csv("../NERC-FixV2W/preprocess/csv_file/2021aug/cwe.csv")
df_cwe.fillna('*', inplace=True)

df_cwe_cate = pd.read_csv("../NERC-FixV2W/preprocess/csv_file/2021aug/cwe_category.csv")
df_cwe_view = pd.read_csv("../NERC-FixV2W/preprocess/csv_file/2021aug/cwe_view.csv")
df_cwe_cate.fillna('*', inplace=True)
df_cwe_view.fillna('*', inplace=True)
cwe_prohibited = df_cwe_cate["ID"].tolist() + df_cwe_view["ID"].tolist()
cwe_disc = ["CWE-119", "CWE-20", "CWE-200", "CWE-269", "CWE-287",
            "CWE-311", "CWE-330", "CWE-345", "CWE-400", "CWE-610",
            "CWE-662", "CWE-665", "CWE-668", "CWE-682", "CWE-697",
            "CWE-74", "CWE-755", "CWE-834"]
cwe_rels = pd.read_csv("../NERC-FixV2W/FixV2W/files/cwe_relations.csv")

cwe_dict = {}
for i, row in cwe_rels.iterrows():
    child = row['Child']
    parent = row['Parent']
    rel = row['Relationship']

    if rel == 'ChildOf':
        if child in cwe_dict:
            cwe_dict[child].append(parent)
        else:
            cwe_dict[child] = []
            cwe_dict[child].append(parent)

In [38]:
def get_sib_fam(cwe):
    family = []
    if cwe in cwe_dict:
        parents = cwe_dict[cwe]
        for p in parents:
            family.extend(get_cwe_children(p))
    return list(set(family))
def get_cwe_children(cwe):
    family = []
    if any(cwe in val for val in cwe_dict.values()):
        keys = [key for key, value in cwe_dict.items() if cwe in value]
        family.extend(keys)
        for item in keys:
            res = get_cwe_children(item)  
            if res:
                family.extend(res)
    return family

In [17]:
curr_disc = []
for i, r in df.iterrows():
    for cwe in r['matching_cwes']:
        if cwe in cwe_disc:
            curr_disc.append([r['cve_id'], cwe])

In [18]:
len(curr_disc)

53019

In [19]:
curr_proh = []
for i, r in df.iterrows():
    for cwe in r['matching_cwes']:
        if cwe in cwe_prohibited:
            curr_proh.append([r['cve_id'], cwe])

In [20]:
len(curr_proh)

13694

In [25]:
df_cwe_1003 = pd.read_csv("../NERC-FixV2W/FixV2W/files/cwe-1003.csv", usecols=['CWE-ID', 'Weakness Abstraction'], index_col=False)
df_cwe_1003.fillna('*', inplace=True)
cwe_1003 = ['CWE-' + str(num) for num in df_cwe_1003['CWE-ID'].values.tolist()]

In [50]:
cwe_test = curr_disc[-1:][0][1]
cve_test = curr_disc[-1:][0][0]
print(curr_disc[-1:][0])

['CVE-2026-29093', 'CWE-668']


In [56]:
candidate_list = [cwe for cwe in get_sib_fam(cwe_test) if cwe in cwe_1003 and cwe not in cwe_disc]
print(len(candidate_list))

59


In [52]:
folder_path = "saved/2026-03-23T15-40-42-551Z/id_conversions"

with open(os.path.join(folder_path, "entity2id.pkl"), "rb") as f:
    entity2id = pickle.load(f)

with open(os.path.join(folder_path, "id2entity.pkl"), "rb") as f:
    id2entity = pickle.load(f)

with open(os.path.join(folder_path, "relation2id.pkl"), "rb") as f:
    relation2id = pickle.load(f)

with open(os.path.join(folder_path, "id2relation.pkl"), "rb") as f:
    id2relation = pickle.load(f)

triples_id = np.load(os.path.join(folder_path, "triples_id.npy"))

In [ ]:
cve_test_id = entity2id[cve_test]
cwe_test_id = entity2id[cwe_test]
candidate_list_id = [entity2id[cwe] for cwe in candidate_list] 

In [63]:
results = []

for cwe_id in tqdm(candidate_list_id):
    # build triples
    pos = np.array([[cve_test_id, relation2id['MatchingCWE'], cwe_id]], dtype=np.int64)
    neg = np.array([[[cve_test_id, relation2id['MatchingCWE'], cwe_id]]], dtype=np.int64)

    # API CALL HERE - externally available
    response = transe_rest_request(pos, neg)

    # extract score
    score = response["outputs"][1]["data"][0]  # pos_score
    results.append((cwe_id, score))
    time.sleep(1)
results_sorted = sorted(results, key=lambda x: x[1], reverse=True)

print("Top 10 CWE candidates:")
for rank, (cwe_id, score) in enumerate(results_sorted[:10], 1):
    print(f"{rank}. {id2entity[cwe_id]} (score={score:.4f})")

  0%|          | 0/59 [00:00<?, ?it/s]

Top 10 CWE candidates:
1. CWE-276 (score=-7.1822)
2. CWE-1321 (score=-8.1516)
3. CWE-601 (score=-8.3549)
4. CWE-209 (score=-8.3649)
5. CWE-384 (score=-8.3734)
6. CWE-681 (score=-8.3829)
7. CWE-669 (score=-8.3998)
8. CWE-787 (score=-8.3999)
9. CWE-494 (score=-8.4435)
10. CWE-613 (score=-8.4606)
